# 06 · Publication figures from derived data

Regenerates the standalone manuscript figures that are not produced inside the analysis
notebooks: **Fig. 1** (study area), **Fig. 2** (VHI series), **Fig. 4** (detection skill),
**Fig. 5** (2024 event) and **Fig. 8** (VHI–SPEI coupling). Figures 3, 6 and 7 are produced by
notebooks 05, 03 and 04 respectively.

Inputs are the processed products in `../data/` — no raw meteorological or yield data are needed.
Run top to bottom; PNG/PDF files are written to `../figures/`.

In [ ]:
import pandas as pd, numpy as np, unicodedata, re
import matplotlib.pyplot as plt, matplotlib as mpl
mpl.rcParams.update({'font.size':9,'axes.linewidth':0.8,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':150,'savefig.dpi':300})
DERIVED='../data/derived/'; SPATIAL='../data/spatial/'; FIGDIR='../figures/'

TR=str.maketrans('ıİşŞçÇğĞöÖüÜ','iissccggoouu')
def fold(s):
    s=str(s).translate(TR); return ''.join(c for c in unicodedata.normalize('NFKD',s) if not unicodedata.combining(c)).lower().strip()

vhi=pd.read_csv(DERIVED+'trakya_district_vhi_vci_tci_monthly.csv'); vhi['VHI']=0.5*vhi['VCI']+0.5*vhi['TCI']; vhi['key']=vhi['name'].map(fold)
S=pd.read_csv(DERIVED+'trakya_spi_spei_1965_2024.csv'); S['key']=S['name'].map(fold)
m=vhi.merge(S.drop(columns='name'),on=['key','year','month'],how='inner')
WIN=[6,7,8,9,10]
print('district VHI:', sorted(vhi.name.unique()))

## Fig. 1 · Study area (7 stations, no locator inset)

In [ ]:
import shapefile
from matplotlib.patches import Polygon as MPoly
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
r=shapefile.Reader(SPATIAL+'Trakya_Merged.shp'); shp=r.shapes()[0]
pts=np.array(shp.points); parts=list(shp.parts)+[len(pts)]
polys=[pts[parts[i]:parts[i+1]] for i in range(len(parts)-1)]
si=pd.read_excel(SPATIAL+'station_info.xlsx'); si.columns=['name','lat_dms','lon_dms','elev']
def dd(s):
    d,mm,sec,h=re.match(r"(\d+)°(\d+)'([\d.]+)\"([NSEW])",s.strip()).groups()
    return (float(d)+float(mm)/60+float(sec)/3600)*(-1 if h in('S','W') else 1)
si['lat']=si.lat_dms.map(dd); si['lon']=si.lon_dms.map(dd)
si=si[si['name']!='Sarıyer'].reset_index(drop=True)
off={'Edirne':(-0.05,0.06),'Uzunköprü':(-0.15,-0.12),'İpsala':(-0.02,-0.15),'Kırklareli':(0.04,0.06),
     'Lüleburgaz':(0.05,-0.11),'Tekirdağ':(0.06,0.05),'Çorlu':(0.06,0.06)}
fig,ax=plt.subplots(figsize=(7.4,5.0))
ax.add_collection(PatchCollection([MPoly(p,closed=True) for p in polys],facecolor='#eaf3e0',edgecolor='#4a7a3a',lw=1.0,alpha=0.9))
for _,s in si.iterrows():
    isv=s['name']=='İpsala'
    ax.plot(s['lon'],s['lat'],'^' if isv else 'o',color='#c0392b' if isv else '#1f3d7a',ms=9 if isv else 8,mec='white',mew=0.8,zorder=5)
    dx,dy=off.get(s['name'],(0.05,0.05))
    ax.annotate(s['name'],(s['lon']+dx,s['lat']+dy),fontsize=8.5,fontweight='bold',color='#c0392b' if isv else '#1a1a1a',ha='left' if dx>0 else 'right',zorder=6)
ax.set_xlim(25.4,29.3); ax.set_ylim(39.6,42.2); ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title('Study area: Turkish Thrace — districts and meteorological stations',fontweight='bold',fontsize=11); ax.grid(True,ls=':',lw=0.4,alpha=0.5)
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor='#1f3d7a',ms=8,label='Rainfed district station'),
                   Line2D([0],[0],marker='^',color='w',markerfacecolor='#c0392b',ms=8,label='İpsala (irrigated valley)')],loc='lower left',fontsize=8)
ax.annotate('N',(29.05,42.0),fontsize=13,fontweight='bold',ha='center'); ax.annotate('↑',(29.05,41.82),fontsize=16,ha='center')
fig.savefig(FIGDIR+'Fig_studyarea.png',bbox_inches='tight'); fig.savefig(FIGDIR+'Fig_studyarea.pdf',bbox_inches='tight'); plt.show()

## Fig. 2 · VHI series (regional line + district heat map)

In [ ]:
disp={'Corlu':'Çorlu','Edirne':'Edirne','Ipsala':'İpsala*','Kirklareli':'Kırklareli','Luleburgaz':'Lüleburgaz','Tekirdag':'Tekirdağ','Uzunkopru':'Uzunköprü'}
order=['Edirne','Kirklareli','Tekirdag','Luleburgaz','Corlu','Uzunkopru','Ipsala']
gs=vhi[vhi.month.isin(WIN)].groupby(['name','year'])['VHI'].mean().reset_index()
piv=gs.pivot(index='name',columns='year',values='VHI').reindex(order); reg=gs.groupby('year')['VHI'].mean()
fig=plt.figure(figsize=(7.2,5.6)); g=fig.add_gridspec(2,1,height_ratios=[1,1.25],hspace=0.35)
axa=fig.add_subplot(g[0]); axa.plot(reg.index,reg.values,'-o',color='#2c3e50',ms=3,lw=1.2)
axa.axhspan(0,20,color='#c0392b',alpha=.08); axa.axhspan(20,40,color='#e67e22',alpha=.07)
axa.axhline(40,color='#e67e22',lw=.7,ls='--'); axa.axhline(20,color='#c0392b',lw=.7,ls='--')
for y in [2001,2007,2024]:
    if y in reg.index: axa.annotate(str(y),(y,reg[y]),textcoords='offset points',xytext=(0,-11),ha='center',fontsize=7,color='#c0392b')
if 2014 in reg.index: axa.annotate('2014',(2014,reg[2014]),textcoords='offset points',xytext=(0,5),ha='center',fontsize=7,color='#27ae60')
axa.set_ylabel('Growing-season VHI'); axa.set_xlim(1984,2025); axa.set_ylim(10,70)
axa.set_title('(a) Regional mean VHI, June–October (α = 0.5)',loc='left',fontweight='bold')
axb=fig.add_subplot(g[1]); im=axb.imshow(piv.values,aspect='auto',cmap='RdYlGn',vmin=15,vmax=65,extent=[1984.5,2024.5,len(order)-0.5,-0.5])
axb.set_yticks(range(len(order))); axb.set_yticklabels([disp[n] for n in order]); axb.set_xlabel('Year')
axb.set_title('(b) District growing-season VHI',loc='left',fontweight='bold'); axb.set_xticks(np.arange(1985,2025,5))
fig.colorbar(im,ax=axb,pad=0.015,fraction=0.045).set_label('VHI',fontsize=8)
fig.text(0.012,0.012,'* İpsala: irrigated rice/valley — VHI does not track rainfed meteorological drought.',fontsize=6.2,color='#7f8c8d',style='italic')
fig.savefig(FIGDIR+'Fig_VHI_series.png',bbox_inches='tight'); fig.savefig(FIGDIR+'Fig_VHI_series.pdf',bbox_inches='tight'); plt.show()

## Fig. 4 · Drought-detection skill (frequency-matched HSS by district)

In [ ]:
d=m[m.month.isin(WIN)].dropna(subset=['VHI','SPEI3'])
def skill(ref,det):
    a=int((ref&det).sum()); b=int((~ref&det).sum()); c=int((ref&~det).sum()); dd=int((~ref&~det).sum()); n=a+b+c+dd
    POD=a/(a+c) if a+c else np.nan; exp=((a+b)*(a+c)+(c+dd)*(b+dd))/n; HSS=(a+dd-exp)/(n-exp) if n-exp else np.nan
    return POD,HSS
disp2={'Corlu':'Çorlu','Edirne':'Edirne','Ipsala':'İpsala','Kirklareli':'Kırklareli','Luleburgaz':'Lüleburgaz','Tekirdag':'Tekirdağ','Uzunkopru':'Uzunköprü'}
rows=[]
for nm,g2 in d.groupby('name'):
    ref=g2['SPEI3']<-1.0
    if len(g2)<25 or ref.sum()<5: continue
    vt=np.quantile(g2['VHI'].values,ref.mean()); _,H=skill(ref,g2['VHI']<vt)
    rows.append({'d':disp2[nm],'HSS':H,'irr':nm=='Ipsala'})
tbl=pd.DataFrame(rows).sort_values('HSS')
fig,ax=plt.subplots(figsize=(6.2,3.4))
ax.barh(tbl['d'],tbl['HSS'],color=['#c0392b' if i else '#2980b9' for i in tbl['irr']])
ax.set_xlabel('Heidke Skill Score (VHI vs SPEI-3 drought detection)')
rmean=np.mean(tbl[~tbl.irr]['HSS']); ax.axvline(rmean,color='#2980b9',ls='--',lw=.8)
ax.text(rmean+0.005,0.2,f'rainfed mean\n{rmean:.2f}',fontsize=7,color='#2980b9')
ax.set_title('Drought detection skill by district',fontweight='bold',fontsize=10,loc='left')
fig.savefig(FIGDIR+'Fig_detection.png',bbox_inches='tight'); fig.savefig(FIGDIR+'Fig_detection.pdf',bbox_inches='tight'); plt.show()

## Fig. 5 · The 2024 drought event

In [ ]:
MONNAME=['J','F','M','A','M','J','J','A','S','O','N','D']
fig,ax=plt.subplots(1,2,figsize=(9.5,3.6))
r24=m[m.year==2024].groupby('month').agg(VHI=('VHI','mean'),SPEI3=('SPEI3','mean'))
axr=ax[0].twinx()
ax[0].plot(r24.index,r24['VHI'],'-o',color='#27ae60',ms=4,lw=1.6,zorder=3)
axr.bar(r24.index,r24['SPEI3'],color=np.where(r24['SPEI3']<0,'#c0392b','#2980b9'),alpha=.35,width=.7)
ax[0].axhline(40,color='#e67e22',ls='--',lw=.7); ax[0].axhline(20,color='#c0392b',ls='--',lw=.7); axr.axhline(0,color='#7f8c8d',lw=.5)
ax[0].set_xticks(range(1,13)); ax[0].set_xticklabels(MONNAME); ax[0].set_ylabel('VHI',color='#27ae60'); axr.set_ylabel('SPEI-3',color='#c0392b')
ax[0].set_ylim(0,70); axr.set_ylim(-3.2,1.5); ax[0].set_xlabel('Month (2024)')
ax[0].set_title('(a) 2024 drought development',loc='left',fontweight='bold',fontsize=10); axr.spines['top'].set_visible(False)
comp=[]
for y,lab in [(2014,'2014\n(wet)'),(2021,'2021\n(normal)'),(2024,'2024\n(drought)')]:
    comp.append((lab,m[(m.year==y)&(m.month.isin(WIN))].groupby('name')['VHI'].mean()))
x=np.arange(3); w=0.11; dists=sorted(comp[0][1].index); colors=plt.cm.tab10(np.linspace(0,1,len(dists)))
dd2={'Corlu':'Çorlu','Edirne':'Edirne','Ipsala':'İpsala','Kirklareli':'Kırklareli','Luleburgaz':'Lüleburgaz','Tekirdag':'Tekirdağ','Uzunkopru':'Uzunköprü'}
for i,dn in enumerate(dists):
    ax[1].bar(x+(i-3)*w,[c[1].get(dn,np.nan) for c in comp],w,label=dd2[dn],color=colors[i])
ax[1].set_xticks(x); ax[1].set_xticklabels([c[0] for c in comp]); ax[1].set_ylabel('Growing-season VHI')
ax[1].axhline(40,color='#e67e22',ls='--',lw=.7); ax[1].set_title('(b) District VHI by year type',loc='left',fontweight='bold',fontsize=10)
ax[1].legend(fontsize=5.5,ncol=2,loc='upper right')
fig.tight_layout(); fig.savefig(FIGDIR+'Fig_case2024.png',bbox_inches='tight'); fig.savefig(FIGDIR+'Fig_case2024.pdf',bbox_inches='tight'); plt.show()

## Fig. 8 · Interannual VHI–SPEI-3 coupling

In [ ]:
from scipy import stats
gsv=m[m.month.isin(WIN)].groupby('year')['VHI'].mean(); sp=S[S.month.isin(WIN)].groupby('year')['SPEI3'].mean()
dd=pd.concat([gsv.rename('VHI'),sp.rename('SPEI3')],axis=1).dropna(); dd=dd[dd.index>=1985]
fig,(a1,a2)=plt.subplots(1,2,figsize=(7.2,3.0),gridspec_kw={'width_ratios':[1.5,1]})
ar=a1.twinx()
a1.plot(dd.index,dd['VHI'],'-o',color='#27ae60',ms=2.5,lw=1); ar.plot(dd.index,dd['SPEI3'],'-s',color='#2980b9',ms=2.5,lw=1,alpha=.8); ar.axhline(0,color='#95a5a6',lw=.5)
a1.set_ylabel('VHI',color='#27ae60'); ar.set_ylabel('SPEI-3',color='#2980b9'); a1.set_xlabel('Year')
a1.set_title('(a) Regional VHI and SPEI-3',loc='left',fontweight='bold'); ar.spines['top'].set_visible(False)
r,p=stats.pearsonr(dd['SPEI3'],dd['VHI'])
a2.scatter(dd['SPEI3'],dd['VHI'],s=18,color='#34495e',alpha=.7,edgecolor='w',linewidth=.3)
b1,b0=np.polyfit(dd['SPEI3'],dd['VHI'],1); xs=np.linspace(dd['SPEI3'].min(),dd['SPEI3'].max(),50); a2.plot(xs,b0+b1*xs,'r-',lw=1.2)
a2.set_xlabel('SPEI-3 (Jun–Oct)'); a2.set_ylabel('VHI'); a2.set_title('(b) Coupling',loc='left',fontweight='bold')
a2.text(0.05,0.92,f'r = {r:.2f}\np < 0.001',transform=a2.transAxes,va='top',fontsize=8,bbox=dict(boxstyle='round',fc='w',ec='#bdc3c7',alpha=.8))
fig.tight_layout(); fig.savefig(FIGDIR+'Fig_VHI_SPEI.png',bbox_inches='tight'); fig.savefig(FIGDIR+'Fig_VHI_SPEI.pdf',bbox_inches='tight'); plt.show()
print('Annual VHI–SPEI-3 r =',round(r,3))

---
**Fig. 3** (drought maps) → notebook `05_vhi_drought_maps.ipynb` (Google Earth Engine).
**Fig. 6** (scale ranking) → notebook `03_scale_selection.ipynb`.
**Fig. 7** (yield panel) → notebook `04_pet_sensitivity_yield_panel.ipynb` (requires the TÜİK yield
file, which is not redistributed here).